# Liberman noise-stratum mean synapse baseline (wide table)

Uses the **Liberman wide** table from `load_nn_stage2_data`: **one row per `animal_id` × `frequency`**, same `DataGroup` train/validate/test split as `abr_nn_stage2` / `splits_for_long_stage2`. **`noise_cat`** is the binary noise label on each wide row (**0** if original `Noise` ≤ 0.91, **1** otherwise), populated when building `reformatted_orig` in `nn_stage2_data` (same rule as `_build_liberman_orig`).

**Aggregate RMSE / R²:** **10-fold `StratifiedGroupKFold` CV** at **`animal_id`** level, stratified by **`experimental_group`** (Liberman **`Group`**). Each fold fits **train-animal** means for **`(noise_cat, frequency)`** (missing strata → global train mean), scores **held-out animals**, and we report **mean ± SEM** across folds (`liberman_group_mean_baseline_metrics.parquet` also stores **`n_cv_folds`**). Matches §5.2 / model prediction strata.

**Per-group table:** **official test split** only, with **`pred`** from the CV fold where that test animal was held out (same stratum-mean rule). **`n_test_animal_freq`** counts test wide rows. **True synapses** mean and SD for each **`noise_cat` × `frequency`** over **all** wide rows. Exported Parquet **`Group`** column equals **`noise_cat`** for deck loaders.

**Working directory:** repository root. Exports → `figures/cache/`.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from sklearn.metrics import r2_score

from utils.nn_stage2_data import load_nn_stage2_data, splits_for_long_stage2
from utils.subject_cv import (
    DEFAULT_CEILING_CV_RANDOM_STATE,
    DEFAULT_CEILING_CV_SPLITS,
    eval_stratum_mean_baseline_cv,
)

In [2]:
data = load_nn_stage2_data()
splits = splits_for_long_stage2(data)

# Wide Liberman rows already carry noise_cat from reformatted_orig (_build_reformatted_orig).
train = splits["lib_train"].copy()
test = splits["lib_test"].copy()
df = data.reformatted_orig.reset_index(drop=True).copy()

len(train), len(test), train[["animal_id", "frequency", "synapses", "noise_cat"]].head(
    2
)

(491,
 125,
   animal_id  frequency   synapses  noise_cat
 0    WPZ101       32.0  16.193497        0.0
 1    WPZ101       45.2  15.351421        0.0)

In [3]:
# Unique animals per noise_cat (wide: multiple rows per animal×frequency, so nunique)
def _animals_per_noise_cat(subset: pd.DataFrame, label: str) -> pd.DataFrame:
    return (
        subset.groupby("noise_cat", dropna=False)["animal_id"]
        .nunique()
        .rename("n_animals")
        .reset_index()
        .assign(split=label)
    )


noise_animal_counts = pd.concat(
    [
        _animals_per_noise_cat(train, "train"),
        _animals_per_noise_cat(test, "test"),
        _animals_per_noise_cat(df, "all"),
    ],
    ignore_index=True,
)
display(noise_animal_counts.sort_values(["noise_cat", "split"]).reset_index(drop=True))
# Wide view: one column per split
display(
    noise_animal_counts.pivot(index="noise_cat", columns="split", values="n_animals")
)

,noise_cat,n_animals,split
0,0.0,50,all
1,0.0,10,test
2,0.0,40,train
3,1.0,55,all
4,1.0,11,test
5,1.0,44,train


split,all,test,train
noise_cat,,,
0.0,50,10,40
1.0,55,11,44


In [4]:
# 10-fold StratifiedGroupKFold CV (stratified by experimental_group): train means per
# (noise_cat, frequency); missing strata → global train mean; OOF preds on every row.
N_FOLDS = DEFAULT_CEILING_CV_SPLITS
RNG = DEFAULT_CEILING_CV_RANDOM_STATE

_cv_out = eval_stratum_mean_baseline_cv(
    df,
    ["noise_cat", "frequency"],
    n_splits=N_FOLDS,
    random_state=RNG,
)
pred_oof = _cv_out["pred_oof"]
r2 = _cv_out["r2"]
r2_sem = _cv_out["r2_sem"]
rmse = _cv_out["rmse"]
rmse_sem = _cv_out["rmse_sem"]
print(
    f"Noise-cat mean baseline (wide) — {N_FOLDS}-fold StratifiedGroupKFold CV: "
    f"R² = {r2:.4f} ± {r2_sem:.4f} (SEM), RMSE = {rmse:.4f} ± {rmse_sem:.4f} (SEM)"
)

Noise-cat mean baseline (wide) — 10-fold StratifiedGroupKFold CV: R² = 0.4241 ± 0.0619 (SEM), RMSE = 2.3420 ± 0.0988 (SEM)


In [5]:
# Official test split: baseline R² / RMSE per noise_cat; preds = OOF CV when each test animal was held out.
_pred_side = df[["animal_id", "frequency"]].copy()
_pred_side["pred"] = pred_oof
test_eval = test.merge(_pred_side, on=["animal_id", "frequency"], how="left")

_rows = []
for nc, part in test_eval.groupby("noise_cat", dropna=False):
    y = part["synapses"].to_numpy(dtype=float)
    p = part["pred"].to_numpy(dtype=float)
    n = int(part.shape[0])
    if n >= 2:
        r2v = float(r2_score(y, p))
    else:
        r2v = float("nan")
    rmse_g = float((np.mean((y - p) ** 2)) ** 0.5) if n else float("nan")
    _rows.append({"Group": nc, "r2": r2v, "rmse": rmse_g, "n_test_animal_freq": n})

group_baseline_metrics_test_df = (
    pd.DataFrame(_rows).sort_values("Group").reset_index(drop=True)
)
print(
    "Test split: per-noise_cat R² / RMSE (pred from 10-fold CV OOF; n_test_animal_freq = test wide rows)"
)
display(group_baseline_metrics_test_df)

Test split: per-noise_cat R² / RMSE (pred from 10-fold CV OOF; n_test_animal_freq = test wide rows)


,Group,r2,rmse,n_test_animal_freq
0,0.0,0.225280,1.520107,60
1,1.0,0.445256,2.895636,65


In [6]:
# All wide rows: mean / std of true synapses per noise_cat × frequency (Group = noise_cat for exports)
group_freq_synapses_df = (
    df.groupby(["noise_cat", "frequency"], dropna=False)["synapses"]
    .agg(mean_synapses="mean", std_synapses="std", n_animal_freq="count")
    .reset_index()
    .rename(columns={"noise_cat": "Group"})
    .sort_values(["Group", "frequency"])
    .reset_index(drop=True)
)
print(
    "True synapses mean / std per noise_cat × frequency (all data); Group = noise_cat"
)
display(group_freq_synapses_df)

True synapses mean / std per noise_cat × frequency (all data); Group = noise_cat


,Group,frequency,mean_synapses,std_synapses,n_animal_freq
0,0.0,8.0,14.865272,1.119300,49
1,0.0,11.3,16.065797,0.779426,49
2,0.0,16.0,17.310863,0.869275,48
3,0.0,22.6,17.422911,1.268345,48
4,0.0,32.0,17.100933,1.931236,50
5,0.0,45.2,16.313863,1.474325,49
6,1.0,8.0,14.556638,1.584356,53
7,1.0,11.3,15.591091,1.312999,54
8,1.0,16.0,16.698790,1.570874,55
9,1.0,22.6,11.337205,3.562464,55


In [7]:
_cache = Path("figures/cache")
_cache.mkdir(parents=True, exist_ok=True)
pd.DataFrame(
    [
        {
            "r2": r2,
            "r2_sem": r2_sem,
            "rmse": rmse,
            "rmse_sem": rmse_sem,
            "n_cv_folds": N_FOLDS,
        }
    ]
).to_parquet(_cache / "liberman_group_mean_baseline_metrics.parquet", index=False)
group_freq_synapses_df.to_parquet(
    _cache / "liberman_synapse_mean_std_by_group_frequency.parquet", index=False
)
group_baseline_metrics_test_df.to_parquet(
    _cache / "liberman_group_baseline_metrics_test_by_group.parquet", index=False
)
print("Saved", (_cache / "liberman_group_mean_baseline_metrics.parquet").resolve())
print(
    "Saved", (_cache / "liberman_synapse_mean_std_by_group_frequency.parquet").resolve()
)
print(
    "Saved",
    (_cache / "liberman_group_baseline_metrics_test_by_group.parquet").resolve(),
)

Saved /Users/nowaki027/MSDS/Practicum/figures/cache/liberman_group_mean_baseline_metrics.parquet
Saved /Users/nowaki027/MSDS/Practicum/figures/cache/liberman_synapse_mean_std_by_group_frequency.parquet
Saved /Users/nowaki027/MSDS/Practicum/figures/cache/liberman_group_baseline_metrics_test_by_group.parquet
